# 0. Environment Setup and Configuration

In [1]:
import sys
from pathlib import Path

_root = Path.cwd().parent
for p in (_root, _root / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from config import (
    USE_LOCAL_EMBEDDINGS, STORAGE_CONN_STR, CONTAINER_NAME,
    LOCAL_FAISS_INDEX_PATH, LOCAL_FAISS_METADATA_PATH,
    CHUNK_SIZE, CHUNK_OVERLAP, MIN_CHUNK_SIZE,
    VIEW_INTERNAL_LOG
)
# show internal log messages from src directory if enabled
import logging
if VIEW_INTERNAL_LOG:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
        force=True,  # important in notebooks so existing handlers are replaced
    )

## 1  Extract
Pull all supported documents from the blob container and inspect the results.

In [2]:
from extract import DocumentExtractor
extractor = DocumentExtractor(
    connection_string=STORAGE_CONN_STR,
    container_name=CONTAINER_NAME,
)
documents = extractor.extract_all()
print(f'Extracted {len(documents)} documents from Azure Blob Storage.')

# Inspect
for doc in documents:
    print(f'  [{doc.category}] {doc.blob_name} ({len(doc.text):,} chars, method={doc.extraction_method})')


2026-03-16 16:27:09,076 [INFO] azure.core.pipeline.policies.http_logging_policy - Request URL: 'https://aidevghazal.blob.core.windows.net/bmo-data-test?restype=REDACTED&comp=REDACTED&prefix=REDACTED'
Request method: 'GET'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.13.2 (macOS-15.7.3-arm64-arm-64bit-Mach-O)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '81ae9fbe-2176-11f1-b5ad-823c494fe81a'
    'Authorization': 'REDACTED'
No body was attached to the request
2026-03-16 16:27:09,267 [INFO] azure.core.pipeline.policies.http_logging_policy - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/xml'
    'x-ms-request-id': '9229bf04-901e-00bc-4d83-b59b05000000'
    'x-ms-client-request-id': '81ae9fbe-2176-11f1-b5ad-823c494fe81a'
    'x-ms-version': 'REDACTED'
    'Date': 'Mon, 16 Mar 2026 20:27:08 GMT'
2026-03-16 16:27:09,272 [IN

Extracted 11 documents from Azure Blob Storage.
  [manuals] manuals/deviceA.pdf (2,859 chars, method=direct)
  [manuals] manuals/deviceB.pdf (3,142 chars, method=direct)
  [manuals] manuals/deviceC.pdf (1,970 chars, method=direct)
  [manuals] manuals/scanned_sample.pdf (1,061 chars, method=ocr)
  [policies] policies/data_retention.txt (3,546 chars, method=direct)
  [policies] policies/security.txt (4,519 chars, method=direct)
  [troubleshooting] troubleshooting/acceptable_use.md (4,160 chars, method=direct)
  [troubleshooting] troubleshooting/deviceD.md (3,858 chars, method=direct)
  [troubleshooting] troubleshooting/error101.md (3,148 chars, method=direct)
  [troubleshooting] troubleshooting/error404.md (2,285 chars, method=direct)
  [troubleshooting] troubleshooting/firmware_recovery.md (2,654 chars, method=direct)


In [4]:
#View first document's metadata and truncated text
for key, value in vars(documents[0]).items():
    if key =="text":
        value = value[:50] + "..." if len(value) > 500 else value  # truncate long text for display
    print(f"{key}: {value}")

blob_name: manuals/deviceA.pdf
source_url: https://aidevghazal.blob.core.windows.net/bmo-data-test/manuals/deviceA.pdf
content_type: pdf
text: Device A — SmartHub Pro User Manual
Model: SHP-300...
page_count: 3
title: Devicea
category: manuals
extracted_at: 2026-03-16T20:27:09.331096+00:00
extraction_method: direct
metadata: {'size_bytes': 5766, 'last_modified': '2026-03-14T21:35:51+00:00', 'content_type': 'application/pdf', 'etag': '"0x8DE8211A9F806F6"'}


## 2  Chunk

In [5]:
from chunk import TextChunker

chunker = TextChunker(chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP, min_chunk_size=MIN_CHUNK_SIZE)  # smaller sizes for demo
chunks = chunker.chunk_documents(documents)

print(f'Total chunks: {len(chunks)}')
print()
for c in chunks[:4]:
    print(f'  chunk_id={c.chunk_id[:8]}…  doc={c.doc_id}  idx={c.chunk_index}/{c.total_chunks-1}  ~{c.token_estimate} tokens')
    print(f'  text: {c.text[:120]}…')
    print()

Total chunks: 38

  chunk_id=e403af04…  doc=devicea  idx=0/3  ~287 tokens
  text: Device A — SmartHub Pro User Manual
Model: SHP-3000 | Firmware: v4.2.1 | Document Rev: 1.5
1. Safety Warnings

  chunk_id=fcf91599…  doc=devicea  idx=1/3  ~129 tokens
  text: the power button on the rear panel for 2 seconds.
4.2 First-Time Configuration
The SmartHub Pro defaults to IP address 1…

  chunk_id=c651a8c4…  doc=devicea  idx=2/3  ~185 tokens
  text: 1G/10G transceivers for fibre uplinks.
6. Wireless Settings
The SmartHub Pro supports dual-band Wi-Fi 6 (802.11ax): 2.4 …

  chunk_id=3c46796e…  doc=devicea  idx=3/3  ~131 tokens
  text: 9. Technical Specifications
Parameter
Value
Processor
Quad-core ARM Cortex-A55 @ 1.8 GHz
RAM
512 MB DDR4
Flash Storage
2…



In [7]:
#View first document's metadata and truncated text
for key, value in vars(chunks[0]).items():
    if key =="text":
        value = value[:50] + "..." if len(value) > 500 else value  # truncate long text for display
    print(f"{key}: {value}")

chunk_id: e403af04-c085-4a3a-bab7-01bd80da6ab7
doc_id: devicea
blob_name: manuals/deviceA.pdf
source_url: https://aidevghazal.blob.core.windows.net/bmo-data-test/manuals/deviceA.pdf
title: Devicea
category: manuals
text: Device A — SmartHub Pro User Manual
Model: SHP-300...
chunk_index: 0
total_chunks: 4
char_start: 0
char_end: 1149
estimated_page: 1
metadata: {'size_bytes': 5766, 'last_modified': '2026-03-14T21:35:51+00:00', 'content_type': 'application/pdf', 'etag': '"0x8DE8211A9F806F6"', 'extraction_method': 'direct'}


## 3  Embed

In [8]:
from embed import EmbeddingClient, LocalEmbeddingClient

if USE_LOCAL_EMBEDDINGS:
    embedder = LocalEmbeddingClient('all-MiniLM-L6-v2')  # 384-dim, fast
else:

    embedder = EmbeddingClient(
        azure_endpoint=OPENAI_ENDPOINT,
        api_key=OPENAI_API_KEY,
        deployment_name=OPENAI_DEPLOYMENT,
        batch_size = BATCH_SIZE,
        retry_delay_seconds = RETRY_DELAY_SECONDS,
    )

chunks = embedder.embed_chunks(chunks)

sample = chunks[0]
emb = sample.metadata['embedding']
print(f'Embedding model : {sample.metadata["embedding_model"]}')
print(f'Embedding dim   : {len(emb)}')
print(f'First 8 values  : {[round(x, 4) for x in emb[:8]]}')

2026-03-16 16:27:46,954 [INFO] sentence_transformers.SentenceTransformer - Use pytorch device_name: mps
2026-03-16 16:27:46,955 [INFO] sentence_transformers.SentenceTransformer - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2026-03-16 16:27:47,103 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 16:27:47,131 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-03-16 16:27:47,187 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 16:27:47,245 [WARNING] huggingface_hub.utils._http - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher r

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-16 16:27:47,982 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 16:27:48,004 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-03-16 16:27:48,058 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 16:27:48,077 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding model : all-MiniLM-L6-v2
Embedding dim   : 384
First 8 values  : [0.0039, 0.064, -0.0263, -0.0479, 0.0579, 0.0414, -0.0431, 0.0764]


## 4  Index Locally with FAISS

This section builds a local FAISS vector index from the embedded chunks.
It replaces Azure AI Search for development and notebook demos.

**Save Indexing**

In [9]:
import importlib
import index as index_module
importlib.reload(index_module)
from index import LocalFaissIndexManager

local_index = LocalFaissIndexManager(
    index_path=LOCAL_FAISS_INDEX_PATH,
    metadata_path=LOCAL_FAISS_METADATA_PATH,
    recreate=True,
)
local_index.build(chunks)
print('Document count in local index:', local_index.get_document_count())

2026-03-16 16:28:00,426 [INFO] faiss.loader - Loading faiss.


2026-03-16 16:28:00,516 [INFO] faiss.loader - Successfully loaded faiss.


Document count in local index: 38


**Load Indexing**

In [10]:
reloaded_index = LocalFaissIndexManager(
    index_path=LOCAL_FAISS_INDEX_PATH,
    metadata_path=LOCAL_FAISS_METADATA_PATH
)
reloaded_index.load()

print('Reloaded document count:', reloaded_index.get_document_count())
print('Index files:', LOCAL_FAISS_INDEX_PATH, LOCAL_FAISS_METADATA_PATH)


2026-03-16 16:28:14,969 [INFO] index - Loaded FAISS index (38 docs).


Reloaded document count: 38
Index files: pipeline_demo_faiss.index pipeline_demo_faiss_metadata.json


## 5  Search
Search can be applied for keyword, hybrid, or vector modes

In [15]:
from search import LocalSearchEngine, SearchMode

local_engine = LocalSearchEngine(
    index_manager=reloaded_index,
    embedder=embedder
 )

queries = [
    'How do I connect CSP-100Z A via Bluetooth?',
    'What is Network Connection Timeout Severity?',
    'What is AI Motion Detection?'
]
# mode = [SearchMode.VECTOR, SearchMode.HYBRID]
mode = SearchMode.HYBRID
for q in queries:
    response = local_engine.search(q, top_k=2, mode=mode)
    print(f'Query: "{q}"')
    print(f'Search mode: {mode}')
    for result in response.results:
        print("-"*40)
        print(f'  {result.rank}. [{result.category}] {result.title}  (score={result.score:.4f})')
        print(f'     {result.text[:150]}')
    print("#"*40)
    print("#"*100)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Query: "How do I connect CSP-100Z A via Bluetooth?"
Search mode: SearchMode.HYBRID
----------------------------------------
  1. [troubleshooting] Deviced  (score=4.5722)
     CSP-100Z is a Zigbee end device and relies on a nearby router for connectivity. Firmware Updates OTA (over-the-air) firmware updates are delivered aut
----------------------------------------
  2. [troubleshooting] Deviced  (score=2.7223)
     Device D — ClimateSensor Pro Quick-Start Guide Model: CSP-100 Firmware: v1.3.2 Document Rev: 1.0 Overview The ClimateSensor Pro is a compact, battery-
########################################
####################################################################################################
Query: "What is Network Connection Timeout Severity?"
Search mode: SearchMode.HYBRID
----------------------------------------
  1. [troubleshooting] Error101  (score=5.0941)
     Error 101: Network Connection Timeout Severity: High Affected Products: SmartHub Pro (SHP-3000), SensorNode 